# Band-specific logistic baselines

## Scientific question

Do compact theta and alpha envelope summaries decode the selected perceptual contrast above chance?

## Model rationale

Regularized logistic regression provides an interpretable, low-variance reference. Theta at FCz and alpha at Oz test two predeclared physiological summaries before using higher-capacity models.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from br_analysis.config import ProjectPaths
from br_analysis.data import load_epoch_dataset
from br_analysis.registry import MODELS, get_model_spec
from br_analysis.scientific_reports import (
    aggregate_metrics,
    cohort_table,
    configuration_table,
    load_saved_run,
    per_subject_metrics,
    plot_class_waveforms,
    plot_confusion,
    plot_prediction_confidence,
    plot_roc_pr_comparison,
    plot_subject_performance,
    prediction_diagnostics,
    subject_trial_table,
)
from br_analysis.tasks import TASKS, get_task

In [ ]:
TASK_ID = "c_vs_ip"
MODEL_IDS = ('lr_theta', 'lr_alpha')
paths = ProjectPaths.from_env("scientific_report", create_dirs=False)
model_specs = [get_model_spec(model_id) for model_id in MODEL_IDS]
primary_model = model_specs[0]
task = get_task(TASK_ID, analysis_window=primary_model.analysis_window)
dataset = load_epoch_dataset(
    paths,
    task,
    primary_model.preprocessing,
    picks=primary_model.channels,
)

In [ ]:
display(Markdown("## Selected task, cohort and configuration"))
for model_spec in model_specs:
    display(configuration_table(task, model_spec))
display(cohort_table(dataset))
subject_counts = subject_trial_table(dataset)
display(subject_counts)
print(f"Included subjects: {len(dataset.subject_ids)} | Trials: {len(dataset.y)}")
if dataset.exclusions:
    print(f"Excluded subjects: {len(dataset.exclusions)}")

In [ ]:
commands = [
    f"br-analysis run --task {TASK_ID} --model {model_id}"
    for model_id in MODEL_IDS
]
command_block = "\n".join(commands)
display(Markdown(f"## Reproducible CLI commands\n\n```powershell\n{command_block}\n```"))

In [ ]:
runs = {
    model_id: load_saved_run(paths.artifacts_dir, TASK_ID, model_id)
    for model_id in MODEL_IDS
}
run_status = pd.DataFrame(
    {
        "model_id": MODEL_IDS,
        "completed": [runs[model_id] is not None for model_id in MODEL_IDS],
        "artifact_directory": [str(paths.artifacts_dir / TASK_ID / model_id) for model_id in MODEL_IDS],
    }
)
display(run_status)

In [ ]:
display(Markdown("## Per-subject and aggregate metrics"))
for model_id, run in runs.items():
    if run is None:
        print(f"{model_id}: no saved run; execute the CLI command above.")
        continue
    display(Markdown(f"### {model_id}"))
    display(aggregate_metrics(run))
    display(per_subject_metrics(run))
    plot_subject_performance(run, chance_level=1 / task.n_classes)
    plt.show()

In [ ]:
display(Markdown("## Confusion matrix and prediction diagnostics"))
for model_id, run in runs.items():
    if run is None:
        continue
    display(Markdown(f"### {model_id}"))
    display(prediction_diagnostics(run))
    plot_confusion(run, task.class_ids)
    plt.show()
    plot_prediction_confidence(run)
    plt.show()

## Cross-run ROC and precision-recall comparisons

These curves pool saved out-of-fold predictions. Binary tasks use the positive class; multiclass tasks use micro-averaged one-vs-rest curves. Compare models within a task directly. Comparisons across tasks are descriptive because class definitions and prevalence differ.

In [ ]:
task_runs = {
    model_id: load_saved_run(paths.artifacts_dir, TASK_ID, model_id)
    for model_id in MODELS
}
plot_roc_pr_comparison(
    task_runs,
    title=f"All completed models for {TASK_ID}",
)
plt.show()
for model_id in MODEL_IDS:
    model_runs = {
        task_id: load_saved_run(paths.artifacts_dir, task_id, model_id)
        for task_id in TASKS
    }
    plot_roc_pr_comparison(
        model_runs,
        title=f"{model_id} across completed tasks",
    )
    plt.show()

## Model-specific visualisation

In [ ]:
for model_spec in model_specs:
    model_task = get_task(TASK_ID, analysis_window=model_spec.analysis_window)
    model_dataset = load_epoch_dataset(
        paths,
        model_task,
        model_spec.preprocessing,
        picks=model_spec.channels,
        verbose=False,
    )
    print(f"{model_spec.id}: {model_dataset.channel_names}")
    plot_class_waveforms(model_dataset)
    plt.show()

## Interpretation, limitations and baseline comparison

Treat these models as reference points rather than mechanistic proof. A strong result can reflect amplitude differences, temporal structure, or residual subject-specific effects. Compare both bands with Riemann and SVC under the same task and grouped folds.